# Range-Doppler Algorithm (RDA) — Time-Domain Stripmap Image Formation

**Objective**: Form a stripmap SAR image from CPHD data using the Range-Doppler Algorithm with subaperture processing and Doppler centroid estimation.

## Overview

The **Range-Doppler Algorithm** is a time-domain SAR image formation technique optimized for stripmap collections. Unlike PFA (frequency-domain), RDA operates on:
1. **Range compression**: Matched filter in range frequency domain
2. **Doppler centroid estimation**: Per-block baseband demodulation
3. **Azimuth compression**: Matched filter in azimuth frequency domain
4. **Mosaicking**: Hann-weighted blending across sub-apertures

## Theory

### RDA Signal Processing Chain

For each range bin:
1. **Range compression**: FFT → multiply by matched filter $H_r(f)$ → IFFT
2. **Doppler centroid ($f_{DC}$)**: Estimate dominant Doppler frequency per block
3. **Baseband demodulation**: $s(t) \rightarrow s(t) e^{-j 2\pi f_{DC} t}$
4. **Azimuth compression**: FFT → multiply by matched filter $H_a(f)$ → IFFT

### Doppler Centroid Estimation

The Doppler centroid is the peak of the azimuth spectrum:
$$f_{DC} = \arg\max_f \left| \sum_t s(t) e^{-j 2\pi f t} \right|$$

This compensates for:
- Platform velocity variations
- Squint angle (non-zero look angle)
- Earth rotation

### Subaperture Processing

Long stripmap collections are partitioned into **azimuth blocks** with overlap:
- Each block has a **local Doppler centroid** $f_{DC,i}$
- Blocks are independently compressed and then mosaicked
- Overlap regions use Hann-weighted averaging

### Modules Used

| Module | Purpose |
|--------|--------|
| `grdl.IO` | CPHD reader via factory (`get_reader('cphd', ...)`) |
| `grdl.image_processing.sar` | `RangeDopplerAlgorithm` |
| `napari` | Interactive visualization |

## Preamble — Version Check

In [ ]:
import grdl
from packaging.version import parse as parse_version

required = "0.6.1"
current = grdl.__version__

if parse_version(current) < parse_version(required):
    raise RuntimeError(
        f"GRDL {required}+ required. Found {current}. "
        f"Update: pip install --upgrade grdl"
    )

print(f"✓ GRDL {current}")

## Imports

In [ ]:
from pathlib import Path

import numpy as np
import napari

from grdl.IO import get_reader
from grdl.image_processing.sar.image_formation import RangeDopplerAlgorithm

## Configuration — User Paths

In [ ]:
# Path to CPHD file (stripmap collection)
CPHD_PATH = Path("/path/to/your/stripmap_cphd.cphd")

# Validate path
assert CPHD_PATH.exists(), f"CPHD file not found: {CPHD_PATH}"
assert CPHD_PATH.suffix.lower() in [".cphd", ".cph"], f"Expected CPHD file, got {CPHD_PATH.suffix}"

print(f"✓ CPHD file: {CPHD_PATH.name}")

## Step 1: Load CPHD Metadata and Signal

In [ ]:
with get_reader('cphd', CPHD_PATH) as reader:
    meta = reader.metadata
    
    # Extract collection parameters
    channel = meta.data.channels[0]
    n_vectors = channel.num_vectors
    n_samples = channel.num_samples
    
    print(f"Collection mode: {meta.data.radar_mode}")
    print(f"Phase history: {n_vectors} vectors × {n_samples} samples")
    print(f"Center frequency: {meta.data.tx_frequency_nominal / 1e9:.2f} GHz")
    
    # Read full signal
    signal = reader.read_signal(channel_id=channel.identifier)

print(f"\nSignal shape: {signal.shape}, dtype: {signal.dtype}")

## Step 2: Configure RDA Parameters

In [ ]:
# RDA algorithm configuration
rda_params = {
    'block_size': 'auto',          # 'auto' = physics-driven sizing, or integer pulse count
    'overlap': 0.5,                # 50% overlap between blocks
    'range_weighting': None,       # None, 'taylor', 'hamming', 'hanning'
    'azimuth_weighting': None,
    'antenna_compensation': False, # Apply antenna pattern correction
}

print("RDA Configuration:")
for k, v in rda_params.items():
    print(f"  {k}: {v}")

## Step 3: Initialize RDA Processor

In [ ]:
with get_reader('cphd', CPHD_PATH) as reader:
    rda = RangeDopplerAlgorithm(
        metadata=reader.metadata,
        **rda_params
    )
    
    # Query block partition
    n_blocks = len(rda.block_indices) if hasattr(rda, 'block_indices') else "single-reference"
    
    print(f"RDA initialized")
    print(f"Azimuth blocks: {n_blocks}")
    print(f"Block sizing: {rda_params['block_size']}")

## Step 4: Run RDA Image Formation

The `RangeDopplerAlgorithm.form()` method:
1. Range compresses entire signal (FFT per pulse)
2. Partitions into azimuth blocks
3. Estimates $f_{DC}$ per block
4. Demodulates to baseband
5. Azimuth compresses each block (FFT per range bin)
6. Mosaics with Hann-weighted blending

In [ ]:
with get_reader('cphd', CPHD_PATH) as reader:
    rda = RangeDopplerAlgorithm(metadata=reader.metadata, **rda_params)
    signal = reader.read_signal()
    
    # Form the stripmap image
    print("Running RDA image formation...")
    image = rda.form(signal)

print(f"✓ Image formed: {image.shape}, dtype: {image.dtype}")

## Step 5: Compute Image Statistics

In [ ]:
magnitude = np.abs(image)
phase = np.angle(image)

print("RDA Image Statistics:")
print(f"  Magnitude — min: {magnitude.min():.2e}, max: {magnitude.max():.2e}")
print(f"  Magnitude — mean: {magnitude.mean():.2e}, std: {magnitude.std():.2e}")

# dB-scale magnitude
magnitude_db = 20 * np.log10(magnitude + 1e-12)
vmax = magnitude_db.max()
vmin = vmax - 50.0  # 50 dB dynamic range

print(f"  dB range: [{vmin:.1f}, {vmax:.1f}] dB")

## Step 6: Visualization — Interactive Napari Viewer

In [ ]:
viewer = napari.Viewer(title="RDA Stripmap Image")

# Layer 1: Magnitude (dB scale)
viewer.add_image(
    magnitude_db,
    name="RDA Magnitude (dB)",
    colormap="gray",
    contrast_limits=[vmin, vmax],
)

# Layer 2: Phase
viewer.add_image(
    phase,
    name="Phase (rad)",
    colormap="twilight",
    visible=False,
)

# Simplify viewer UI - hide unnecessary controls
viewer.window._qt_viewer.dockLayerControls.setVisible(False)  # Hide layer controls dock
viewer.window._qt_viewer.dockConsole.setVisible(False)  # Hide console
try:
    viewer.window._qt_viewer.activityDock.setVisible(False)  # Hide activity dock if present
except AttributeError:
    pass

print("✓ Napari viewer launched")
print("  RDA operates in range-Doppler domain (time-domain azimuth)")
print("  Compare with PFA frequency-domain formation")

## Physical Interpretation

### RDA vs PFA

| Aspect | RDA | PFA |
|--------|-----|-----|
| **Domain** | Time (azimuth), Freq (range) | Frequency (both) |
| **Doppler centroid** | Explicitly estimated & removed | Implicit in polar grid |
| **Interpolation** | Minimal (only for windowing) | Heavy (polar → rectangular) |
| **Efficiency** | Faster for stripmap | Faster for spotlight |
| **Accuracy** | Exact for stripmap geometry | Approximation (far-field) |

### Block Size Selection
- **`block_size='auto'`**: Physics-driven sizing based on aperture coherence
- **Fixed integer**: Manual control (e.g., 8000 pulses)
- **`block_size=0`**: Single-reference mode (no subapertures, fastest)

### Weighting Windows
- **Taylor**: Optimal sidelobe suppression (best resolution)
- **Hamming/Hanning**: Moderate sidelobes, wider mainlobe
- **Uniform** (None): No windowing (narrowest mainlobe, highest sidelobes)

### Antenna Pattern Compensation
When `antenna_compensation=True`, RDA applies:
$$s_{\text{corrected}}(t) = \frac{s(t)}{A(\theta(t))}$$
where $A(\theta)$ is the azimuth antenna gain. This equalizes backscatter across the strip.

## Summary

Successfully demonstrated:
- ✅ RDA stripmap image formation from CPHD
- ✅ Automatic Doppler centroid estimation per block
- ✅ Subaperture processing with configurable block size
- ✅ Range and azimuth matched filtering
- ✅ Interactive napari visualization

### Key GRDL Patterns
- `get_reader('cphd', path)` for stripmap CPHD
- `RangeDopplerAlgorithm(metadata, block_size='auto', overlap=0.5)`
- Single `form(signal)` call handles full RDA pipeline
- Physics-driven auto-sizing vs manual block control

### Next Steps
- Try `block_size=0` for single-reference mode (fastest, no mosaicking)
- Enable `antenna_compensation=True` for gain equalization
- Compare with Stripmap PFA (frequency-domain, previous notebook)
- Apply range/azimuth weighting (`'taylor'`) for sidelobe suppression
- Export to SICD: `SICDWriter.write(image, metadata, output_path)`